In [1]:
!pip install timm torch torchvision pandas scikit-learn tqdm


In [ ]:
import os
import pandas as pd
import torch
from PIL import Image

import torch.nn as nn
from torchvision import transforms
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset


from sklearn.model_selection import train_test_split
import timm

from sklearn.metrics import accuracy_score

from tqdm import tqdm
from typing import Optional, List, Tuple, Any, Union

In [ ]:
class CustomImageDataset(Dataset):
    """
    Кастомный датасет для загрузки изображений с поддержкой train/test режимов.
    
    Поддерживает:
    - Загрузку изображений с метками классов для обучения
    - Загрузку тестовых изображений без меток (фейковые метки)
    - Обработку битых изображений (замену на белый квадрат)
    - Применение трансформаций к изображениям
    
    Атрибуты:
    ----------
    root_dir : str
        Путь к корневой директории с данными
    mode : str, optional
        Режим работы ("train" или "test"), по умолчанию "train"
    transform : Optional[Any], optional
        Трансформации для применения к изображениям, по умолчанию None
    """
    
    def __init__(self, 
                 root_dir: str, 
                 mode: str = "train", 
                 transform: Optional[Any] = None) -> None:
        """
        Инициализация датасета.
        
        Параметры:
        ----------
        root_dir : str
            Путь к корневой директории с изображениями
        mode : str, optional
            Режим работы ("train" или "test"), по умолчанию "train"
        transform : Optional[Any], optional
            Трансформации torchvision для изображений, по умолчанию None
        """
        self.root_dir = root_dir
        self.mode = mode
        self.transform = transform
        self.image_paths: List[str] = []
        self.labels: List[int] = []
        
        self._load_data()

    def _load_data(self) -> None:
        """Загружает пути к изображениям и метки в зависимости от режима."""
        if self.mode == "train":
            self._load_train_data()
        elif self.mode == "test":
            self._load_test_data()

    def _load_train_data(self) -> None:
        """Загружает обучающие данные с метками классов."""
        for label in ['0', '1']:
            label_dir = os.path.join(self.root_dir, label)
            if not os.path.exists(label_dir):
                continue
                
            for fname in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, fname))
                self.labels.append(int(label))

    def _load_test_data(self) -> None:
        """Загружает тестовые данные с фейковыми метками."""
        df = pd.read_csv(os.path.join(self.root_dir, "test.csv"))
        self.image_paths = [
            os.path.join(self.root_dir, "test_images", fname) 
            for fname in df['id']
        ]
        self.labels = [-1] * len(self.image_paths)  # Фейковые метки для теста

    def __len__(self) -> int:
        """
        Возвращает количество элементов в датасете.
        
        Возвращает:
        -----------
        int
            Количество изображений в датасете
        """
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, str]:
        """
        Возвращает элемент датасета по индексу.
        
        Параметры:
        ----------
        idx : int
            Индекс элемента
            
        Возвращает:
        -----------
        Tuple[torch.Tensor, int, str]
            Кортеж содержащий:
            - Изображение после трансформаций
            - Метку класса
            - Имя файла изображения
        """
        img_path = self.image_paths[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
        except (IOError, OSError):
            # Создаем белое изображение при ошибке загрузки
            image = Image.new("RGB", (224, 224), color=(255, 255, 255))
            
        if self.transform:
            image = self.transform(image)
            
        label = self.labels[idx]
        filename = os.path.basename(img_path)
        
        return image, label, filename

In [ ]:
# Аугментации
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Датасет
full_dataset = CustomImageDataset("/kaggle/input/ghdxfgh/train/train", mode="train", transform=transform_train)

# Трейн/валидация
indices = list(range(len(full_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=full_dataset.labels)

train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=32, shuffle=True)
val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=32, shuffle=False)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

model = timm.create_model("efficientnet_b0", pretrained=True)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, 1)
)
model = model.to(device)


cuda


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):
    # === Обучение ===
    model.train()
    total_loss = 0
    for imgs, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1} [train]"):
        imgs, labels = imgs.to(device), labels.float().to(device).unsqueeze(1)
        preds = model(imgs)
        loss = criterion(preds, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"📚 Epoch {epoch+1}: Train loss = {avg_train_loss:.4f}")

    # === Валидация ===
    model.eval()
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for imgs, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1} [val]"):
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.sigmoid(outputs).squeeze(1).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            val_preds.extend(preds)
            val_labels.extend(labels.numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    print(f"🧪 Epoch {epoch+1}: Val accuracy = {val_acc:.4f}")


Epoch 1 [train]: 100%|██████████| 169/169 [01:38<00:00,  1.71it/s]


📚 Epoch 1: Train loss = 0.0163


Epoch 1 [val]: 100%|██████████| 43/43 [00:21<00:00,  2.01it/s]


🧪 Epoch 1: Val accuracy = 0.9460


Epoch 2 [train]: 100%|██████████| 169/169 [01:29<00:00,  1.88it/s]


...

🧪 Epoch 9: Val accuracy = 0.9526


Epoch 10 [train]: 100%|██████████| 169/169 [01:30<00:00,  1.87it/s]


📚 Epoch 10: Train loss = 0.0075


Epoch 10 [val]: 100%|██████████| 43/43 [00:19<00:00,  2.19it/s]

🧪 Epoch 10: Val accuracy = 0.9489


In [ ]:
# Параметры
BATCH_SIZE = 32
THRESHOLD = 0.5  # можно потом подобрать по валидации

# Подгружаем тестовые данные
test_dataset = CustomImageDataset(root_dir="/kaggle/input/ghdxfgh/test/test", mode="test", transform=transform_val)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Предсказания
model.eval()
results = []

with torch.no_grad():
    for imgs, _, names in tqdm(test_loader, desc="🔮 Inference on test"):
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.sigmoid(outputs).squeeze(1).cpu().numpy()
        preds = (probs > THRESHOLD).astype(int)
        results.extend(zip(names, preds))

# Загружаем порядок из test.csv
test_order = pd.read_csv("/kaggle/input/ghdxfgh/test.csv")

# Собираем финальный submission.csv
result_df = pd.DataFrame(results, columns=["id", "Category"])
submission_df = test_order.merge(result_df, on="id", how="left")

# Обработка отсутствующих файлов: ставим им класс 1
missing_mask = submission_df["Category"].isna()
if missing_mask.any():
    print(f"Внимание: не найдено {missing_mask.sum()} изображений, для них ставим Category = 1")
    submission_df.loc[missing_mask, "Category"] = 1

# Сохраняем
submission_df[["Category"]].to_csv("submission_effnet2.csv", index=False)
print("Готово! submission.csv сохранён.")


🔮 Inference on test: 100%|██████████| 65/65 [00:38<00:00,  1.68it/s]

✅ Готово! submission.csv сохранён.


In [13]:
len(pd.read_csv("/kaggle/working/submission_effnet.csv"))

2064